In [ ]:
# In PostgreSQL, create a database named "poc_to_prod" and a user named "XYZ" with password "********". 
# Grant all privileges on the database to the user.

In [2]:
# Create a new schema with the following SQL command:
# CREATE SCHEMA IF NOT EXISTS memory;

In [3]:
# Enable the extensions in the "poc_to_prod" database:

# CREATE EXTENSION IF NOT EXISTS vector;
# CREATE EXTENSION IF NOT EXISTS "uuid-ossp";
# CREATE EXTENSION IF NOT EXISTS "pg_trgm";


In [ ]:
# While creating the table, the very basic columns of the table would be as follows:
# MESSAGE_ID 
# SENDER (HUMAN OR ASSISTANT)
# MESSAGE 
# CREATED_AT (TIMESTAMP)

In [ ]:
# If I only add these columns, then I would not be able to do a lot of things. 
# For example, I would not be able to get messages only for a specific user. Obviously i dont want users to see each other's messages.
# So I would need to add a USER_ID column to the table.
# 
# The same user might have multiple conversations with the assistant. 
# So I would need to add a SESSION_ID column to the table and manage the conversations like sessions.
# 
# From this, we have understood that the relational design of the memory would have user_id & session_id along with message_id.


In [ ]:
# But what else should we include in the memory?
# What should our memory design look like?
# What are the cases where this simplistic design would fail?

In [ ]:
# Schema Design :

# users -> user_id (PK), name, email, password, created_at, updated_at, last_login_at
# sessions -> session_id (PK), user_id (FK), session_name, created_at, terminated_at
# messages -> message_id (PK), session_id (FK), sender, message, created_at

# This design allows us to manage users, their sessions, and the messages within those sessions.
# Let us proceed with this design and implement it in our database.

In [ ]:
# memory.users
"""
CREATE TABLE MEMORY.USERS (
    user_id UUID PRIMARY KEY DEFAULT UUID_GENERATE_V4(),
    name VARCHAR(255),
    email VARCHAR(255),
    password TEXT NOT NULL,
    created_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,
    last_login_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
);
"""

In [ ]:
# memory.sessions
"""
CREATE TABLE MEMORY.SESSIONS (
    session_id UUID PRIMARY KEY DEFAULT UUID_GENERATE_V4(),
    user_id UUID NOT NULL,
    session_name VARCHAR(60),
    is_active BOOLEAN DEFAULT TRUE,
    created_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,
    terminated_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
);
"""

In [ ]:
# memory.chats
"""
CREATE TABLE MEMORY.CHATS (
    chat_id UUID PRIMARY KEY DEFAULT UUID_GENERATE_V4(),
    session_id UUID NOT NULL,
    sender UUID NOT NULL,
    message TEXT NOT NULL,
    created_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
);
"""

In [18]:
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "poc_to_prod",
    "user": "postgres",
    "password": "**********"
}

In [19]:
import psycopg2
from psycopg2.extras import RealDictCursor

In [20]:
conn = psycopg2.connect(**DB_CONFIG)
conn.autocommit = False
cur = conn.cursor(cursor_factory=RealDictCursor)
print("Connected:", conn.status)  # 1 = CONNECTION_OK

Connected: 1


In [ ]:
# In case you got error while connecting for authentication,
# just check username and password.
# You will get the username from PGadmin dashboard and password is the one that you have set while creating the user in PostgreSQL.
# You can also update password for the user in PGadmin dashboard if you have forgotten it with this query:
# ALTER USER postgres WITH PASSWORD '**********'; -- replace postgres with your username and ********** with your new password.

In [22]:
query = "SELECT * FROM MEMORY.USERS;"
cur.execute(query)
users = cur.fetchall()

In [23]:
users

[]

In [24]:
# Don't forget to close the connection after you're done. This is important to free up resources and avoid potential memory leaks.
cur.close()
conn.close()

In [25]:
# Let us write following codes :
# 1. Create a new user in the database.
# 2. Create a new session for that user.
# 3. Add a few messages to that session.
# 4. Fetch all / "N" messages for that session. This will be the conversation history that we will use as memory for the assistant to refer to while answering the user's queries.

# Basically we need to implement the Create (INSERT) and Read (SELECT) operations for the three tables that we have created. 
# We can implement Update and Delete operations as well but for now, let's focus on Create and Read operations.

In [26]:
# 1. Create a new user in the database.

query_create_user = """
INSERT INTO MEMORY.USERS (name, email, password)
VALUES (%s, %s, %s)
RETURNING user_id, name, email, created_at;
"""

In [31]:
def create_user(name, email, password):
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    cur.execute(query_create_user, (name, email, password))
    new_user = cur.fetchone()
    conn.commit()  # Commit the transaction to save changes to the database
    cur.close()
    conn.close()
    return new_user

In [32]:
# As per data governance and security best practices, we should not store plain text passwords in the database.
# We should hash the passwords before storing them. We can use libraries like bcrypt or argon2 for hashing passwords in Python.
# For example, we can use bcrypt library to hash the password before storing it in the database. Here is how you can do it:

import bcrypt

def hash_password(plain_password):
    # Generate a salt and hash the password
    salt = bcrypt.gensalt()
    hashed_password = bcrypt.hashpw(plain_password.encode('utf-8'), salt)
    return hashed_password.decode('utf-8')  # Store as string in the database

In [33]:
# Create a new user with hashed password

create_user("Pritesh Jha", "pritesh-jha@email.com", hash_password("my_secure_password"))

RealDictRow([('user_id', 'a93b2c00-d4f2-4fdd-8ecd-ae4bfe1b541d'),
             ('name', 'Pritesh Jha'),
             ('email', 'pritesh-jha@email.com'),
             ('created_at',
              datetime.datetime(2026, 3, 7, 21, 53, 33, 820190, tzinfo=datetime.timezone(datetime.timedelta(seconds=19800))))])

* Check in pgadmin console. The new user must be added there.

![added_new_user.png](../assets/snapshots/added_new_user.png)